# 🚀 Mini Project — Pipeline Penjualan Q2 + Data Pelanggan
**Waktu:** ±60 menit (berkelompok 2–3 orang) · **Presentasi:** 3 menit/kelompok

## Pertanyaan bisnis
> *Segmen pelanggan mana yang paling menguntungkan di Q2 2026, bagaimana trennya per bulan, dan siapa pelanggan terbesar?*

## Sumber data
| Data | API | CSV cadangan |
|---|---|---|
| Transaksi Q2 | `api/static/sales_q2_page_{page}.json` | `data/sales_q2_raw.csv` |
| Pelanggan | `api/static/customers.json` | `data/customers_raw.csv` |

## Penilaian
| Aspek | Bobot |
|---|---|
| Pipeline berjalan end-to-end (Extract → Load) | 30% |
| Kualitas data cleaning (lolos semua cek) | 25% |
| Ketepatan query SQL | 20% |
| Dashboard & insight | 25% |

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [ ]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

In [ ]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

In [ ]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

In [ ]:
def run_sql(query):
    """Menjalankan query SELECT dan mengembalikan DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [ ]:
if "{page}" in API_URL:   # mode static JSON
    API_Q2 = API_URL.replace("sales_page_{page}", "sales_q2_page_{page}")
    API_CUSTOMERS = API_URL.replace("sales_page_{page}", "customers")
else:                     # mode API server
    API_Q2 = API_URL.rsplit("/sales", 1)[0] + "/sales-q2"
    API_CUSTOMERS = API_URL.rsplit("/sales", 1)[0] + "/customers"
print(API_Q2, API_CUSTOMERS, sep="\n")

## Tahap 1 — Extract
Ambil transaksi Q2 (berhalaman) dan data pelanggan (satu respons). Gunakan fallback CSV jika perlu.

In [ ]:
# TODO: df_sales_raw = ...
# TODO: df_cust_raw = ...

## Tahap 2 — Transform
- Transaksi: aturan yang sama dengan Exercise 2
- Pelanggan: rapikan nama, standarkan `segment` menjadi `Retail`/`SME`/`Corporate`, konversi `join_date`, hapus duplikat `customer_id`
- Gabungkan (`merge`) transaksi dengan pelanggan → `df_enriched`

In [ ]:
# TODO: df_sales = ...
# TODO: df_cust = ...
# TODO: df_enriched = ...

In [ ]:
assert len(df_sales) == 260, "transaksi Q2 bersih harus 260 baris"
assert df_cust["customer_id"].is_unique and len(df_cust) == 60, "pelanggan unik harus 60"
assert set(df_cust["segment"]) == {"Retail", "SME", "Corporate"}
assert len(df_enriched) == 260 and df_enriched["segment"].notna().all()
print("✅ Transform lolos")

## Tahap 3 — Load
Muat `df_enriched` ke tabel `sales_q2_enriched` di schema Anda.

In [ ]:
# TODO

## Tahap 4 — SQL
Jawab dengan query: (a) revenue & jumlah transaksi per segmen, (b) revenue per segmen per bulan, (c) top 5 pelanggan beserta nama & segmennya.

In [ ]:
# TODO
run_sql(f"SELECT 1")

## Tahap 5 — Dashboard
Buat minimal 3 visual (Plotly atau Looker Studio).

In [ ]:
# TODO

## Tahap 6 — Insight & Rekomendasi
Tuliskan 3 temuan utama dan 1 rekomendasi bisnis.

1. ...
2. ...
3. ...

**Rekomendasi:** ...

## 📤 Pengumpulan
1. Simpan notebook (`Ctrl+S`), lalu di panel **Source Control** VS Code: stage → commit → **Sync Changes/Push** ke fork Anda
2. Isi juga `submissions/NAMA_KELOMPOK.md` di fork Anda (template di folder `submissions/`)
3. Kirim link fork Anda ke form pengumpulan dari panitia